# Potato Disease Classification ? Corrected Production-Candidate Pipeline

This notebook preserves **EfficientNet-B0** while correcting the main training and evaluation risks in the legacy Potato notebook.

Key guarantees:

- portable configuration instead of hardcoded machine paths;
- deterministic execution and worker seeding;
- persisted stratified 70/15/15 split manifest;
- independent train/validation/test transform instances;
- no class-specific augmentation or unvalidated regularization stack;
- frozen BatchNorm statistics during frozen-backbone training;
- one global validation **macro-F1** checkpoint across both phases;
- single-pass production evaluation, temperature calibration and uncertainty rejection;
- Grad-CAM++ retained only as explanation, not as validated severity.

The legacy notebook and checkpoints remain unchanged under `crops/Potato/`.


## 1. Imports, configuration and deterministic runtime

Set `AGRIMINDS_DATA_ROOT` to the directory containing the crop folders. The notebook never embeds a developer-specific absolute path.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import random
import shutil
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import ImageFolder
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "crops").exists() and (candidate / "Notebooks").exists():
            return candidate
    raise FileNotFoundError("Could not locate the AgriMinds project root from the current directory.")


def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_ROOT = Path(os.environ.get("AGRIMINDS_DATA_ROOT", PROJECT_ROOT / "Data" / "prepro_pbl_dataset"))
DATASET_PATH = DATA_ROOT / "potato"
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts" / "potato"
SPLIT_DIR = PROJECT_ROOT / "artifacts" / "splits"
MODEL_DIR = ARTIFACT_ROOT / "models"
REPORT_DIR = ARTIFACT_ROOT / "reports"

SEED = 42
BATCH_SIZE = 32
NUM_WORKERS = 0 if os.name == "nt" else 4
IMAGE_SIZE = 224
ARCHITECTURE = "EfficientNetB0"
EXPECTED_CLASS_TO_IDX = {
    "Potato___Early_blight": 0,
    "Potato___Late_blight": 1,
    "Potato___healthy": 2,
}
EXPECTED_CLASS_NAMES = list(EXPECTED_CLASS_TO_IDX)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = DEVICE.type == "cuda"

for directory in (SPLIT_DIR, MODEL_DIR, REPORT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

seed_everything(SEED)

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Potato dataset not found at {DATASET_PATH}. "
        "Set AGRIMINDS_DATA_ROOT to the directory containing the crop folders."
    )

print(f"Project root : {PROJECT_ROOT}")
print(f"Dataset      : {DATASET_PATH}")
print(f"Artifacts    : {ARTIFACT_ROOT}")
print(f"Device       : {DEVICE}")
print(f"Seed         : {SEED}")


## 2. Android-compatible preprocessing

Validation, testing and production inference use exactly one deterministic resize/normalize pass. Training augmentation is intentionally conservative for the first corrected baseline.


In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.RandomResizedCrop(IMAGE_SIZE, scale=(0.85, 1.0), ratio=(0.90, 1.10)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=12),
    T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

eval_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

PREPROCESS_SPEC = {
    "image_size": [IMAGE_SIZE, IMAGE_SIZE],
    "color_mode": "RGB",
    "resize": "bilinear",
    "mean": MEAN,
    "std": STD,
    "production_tta": False,
}

print("Transforms and production preprocessing specification are ready.")


## 3. Persisted stratified split manifest

The manifest stores relative paths rather than fragile numeric indices. Re-running the notebook verifies the dataset signature and reuses the same split.


In [ ]:
metadata_dataset = ImageFolder(str(DATASET_PATH))
if metadata_dataset.class_to_idx != EXPECTED_CLASS_TO_IDX:
    raise RuntimeError(
        f"Potato class mapping mismatch. Expected {EXPECTED_CLASS_TO_IDX}, "
        f"found {metadata_dataset.class_to_idx}."
    )
class_names = EXPECTED_CLASS_NAMES.copy()
num_classes = len(class_names)
relative_samples = [Path(path).relative_to(DATASET_PATH).as_posix() for path, _ in metadata_dataset.samples]
labels = np.asarray(metadata_dataset.targets, dtype=np.int64)


def dataset_signature(paths, targets, class_to_idx):
    digest = hashlib.sha256()
    digest.update(json.dumps(class_to_idx, sort_keys=True).encode("utf-8"))
    for path, label in zip(paths, targets):
        digest.update(f"{path}\t{int(label)}\n".encode("utf-8"))
    return digest.hexdigest()


DATASET_SIGNATURE = dataset_signature(relative_samples, labels, metadata_dataset.class_to_idx)
SPLIT_MANIFEST = SPLIT_DIR / f"potato_seed{SEED}.json"


def create_split_manifest():
    indices = np.arange(len(labels))
    train_idx, temporary_idx = train_test_split(
        indices,
        test_size=0.30,
        stratify=labels,
        random_state=SEED,
    )
    val_idx, test_idx = train_test_split(
        temporary_idx,
        test_size=0.50,
        stratify=labels[temporary_idx],
        random_state=SEED,
    )
    return {
        "crop": "potato",
        "seed": SEED,
        "dataset_signature": DATASET_SIGNATURE,
        "class_to_idx": metadata_dataset.class_to_idx,
        "splits": {
            "train": [relative_samples[i] for i in train_idx],
            "validation": [relative_samples[i] for i in val_idx],
            "test": [relative_samples[i] for i in test_idx],
        },
    }


if SPLIT_MANIFEST.exists():
    split_payload = json.loads(SPLIT_MANIFEST.read_text(encoding="utf-8"))
    if split_payload["dataset_signature"] != DATASET_SIGNATURE:
        raise RuntimeError(
            "Dataset signature changed after the split was created. "
            "Audit the dataset before creating a new manifest."
        )
    if split_payload["class_to_idx"] != metadata_dataset.class_to_idx:
        raise RuntimeError("Class ordering changed after the split was created.")
else:
    split_payload = create_split_manifest()
    SPLIT_MANIFEST.write_text(json.dumps(split_payload, indent=2), encoding="utf-8")

path_to_index = {path: index for index, path in enumerate(relative_samples)}
train_indices = [path_to_index[path] for path in split_payload["splits"]["train"]]
val_indices = [path_to_index[path] for path in split_payload["splits"]["validation"]]
test_indices = [path_to_index[path] for path in split_payload["splits"]["test"]]

split_sets = [set(train_indices), set(val_indices), set(test_indices)]
assert not (split_sets[0] & split_sets[1])
assert not (split_sets[0] & split_sets[2])
assert not (split_sets[1] & split_sets[2])
assert sum(map(len, split_sets)) == len(metadata_dataset)

train_base = ImageFolder(str(DATASET_PATH), transform=train_transform)
val_base = ImageFolder(str(DATASET_PATH), transform=eval_transform)
test_base = ImageFolder(str(DATASET_PATH), transform=eval_transform)

assert [path for path, _ in train_base.samples] == [path for path, _ in metadata_dataset.samples]
assert [path for path, _ in val_base.samples] == [path for path, _ in metadata_dataset.samples]
assert [path for path, _ in test_base.samples] == [path for path, _ in metadata_dataset.samples]

train_dataset = Subset(train_base, train_indices)
val_dataset = Subset(val_base, val_indices)
test_dataset = Subset(test_base, test_indices)


def seed_worker(worker_id):
    worker_seed = (SEED + worker_id) % (2**32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)


def make_loader(dataset, shuffle):
    generator = torch.Generator().manual_seed(SEED)
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        worker_init_fn=seed_worker,
        generator=generator,
        persistent_workers=NUM_WORKERS > 0,
    )


train_loader = make_loader(train_dataset, shuffle=True)
val_loader = make_loader(val_dataset, shuffle=False)
test_loader = make_loader(test_dataset, shuffle=False)

class_map_path = ARTIFACT_ROOT / "potato_classes.json"
class_map_path.write_text(json.dumps(class_names, indent=2), encoding="utf-8")

print(f"Classes       : {class_names}")
print(f"Train / Val / Test: {len(train_indices)} / {len(val_indices)} / {len(test_indices)}")
print(f"Split manifest: {SPLIT_MANIFEST}")
print(f"Dataset signature: {DATASET_SIGNATURE[:16]}...")


## 4. Class-weighted objective

The corrected baseline uses one imbalance intervention: class-weighted cross-entropy. Label smoothing, MixUp, CutMix and weighted sampling must be tested later as controlled ablations rather than stacked automatically.


In [ ]:
train_labels = labels[train_indices]
class_counts = np.bincount(train_labels, minlength=num_classes)
if np.any(class_counts == 0):
    raise RuntimeError(f"At least one class is absent from the training split: {class_counts.tolist()}")

class_weights = len(train_labels) / (num_classes * class_counts.astype(np.float64))
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

for name, count, weight in zip(class_names, class_counts, class_weights):
    print(f"{name:28s} count={count:4d} weight={weight:.4f}")


## 5. EfficientNet-B0 and frozen BatchNorm policy

The architecture remains unchanged. Frozen feature parameters and their BatchNorm running statistics remain frozen during phase one.


In [ ]:
def build_model(num_classes):
    model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model


def freeze_backbone(model):
    for parameter in model.features.parameters():
        parameter.requires_grad = False
    for parameter in model.classifier.parameters():
        parameter.requires_grad = True


def unfreeze_final_blocks(model, block_count=2):
    for parameter in model.parameters():
        parameter.requires_grad = False
    for block in model.features[-block_count:]:
        for parameter in block.parameters():
            parameter.requires_grad = True
    for parameter in model.classifier.parameters():
        parameter.requires_grad = True


def set_frozen_batchnorm_eval(model):
    for module in model.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            parameters = list(module.parameters(recurse=False))
            if parameters and not any(parameter.requires_grad for parameter in parameters):
                module.eval()


def parameter_summary(model):
    total = sum(parameter.numel() for parameter in model.parameters())
    trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    return total, trainable


model = build_model(num_classes).to(DEVICE)
freeze_backbone(model)
total_params, trainable_params = parameter_summary(model)

print(f"Architecture      : {ARCHITECTURE}")
print(f"Total parameters  : {total_params:,}")
print(f"Trainable phase 1 : {trainable_params:,}")


## 6. Training, validation metrics and global checkpointing

Checkpoints are selected by validation macro-F1. The best score is carried from phase one into phase two.


In [ ]:
PHASE1_PATH = MODEL_DIR / "potato_phase1_corrected.pt"
CHAMPION_PATH = MODEL_DIR / "potato_champion.pt"


def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    if training:
        set_frozen_batchnorm_eval(model)

    running_loss = 0.0
    predictions, targets = [], []
    start = time.perf_counter()

    context = torch.enable_grad() if training else torch.inference_mode()
    with context:
        for images, batch_targets in loader:
            images = images.to(DEVICE, non_blocking=True)
            batch_targets = batch_targets.to(DEVICE, non_blocking=True)

            if training:
                optimizer.zero_grad(set_to_none=True)

            logits = model(images)
            loss = criterion(logits, batch_targets)

            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    [parameter for parameter in model.parameters() if parameter.requires_grad],
                    max_norm=5.0,
                )
                optimizer.step()

            running_loss += loss.item() * images.size(0)
            predictions.extend(logits.argmax(dim=1).detach().cpu().tolist())
            targets.extend(batch_targets.detach().cpu().tolist())

    elapsed = time.perf_counter() - start
    return {
        "loss": running_loss / len(loader.dataset),
        "accuracy": accuracy_score(targets, predictions),
        "balanced_accuracy": balanced_accuracy_score(targets, predictions),
        "macro_f1": f1_score(targets, predictions, average="macro", zero_division=0),
        "seconds": elapsed,
    }


def checkpoint_payload(model, phase, epoch, best_macro_f1):
    return {
        "model_state": model.state_dict(),
        "architecture": ARCHITECTURE,
        "class_names": class_names,
        "class_to_idx": metadata_dataset.class_to_idx,
        "preprocess": PREPROCESS_SPEC,
        "dataset_signature": DATASET_SIGNATURE,
        "split_manifest": SPLIT_MANIFEST.name,
        "seed": SEED,
        "phase": phase,
        "epoch": epoch,
        "best_val_macro_f1": float(best_macro_f1),
    }


def load_checkpoint(path, model):
    payload = torch.load(path, map_location=DEVICE, weights_only=True)
    if payload["architecture"] != ARCHITECTURE:
        raise RuntimeError("Checkpoint architecture mismatch.")
    if payload["class_names"] != class_names:
        raise RuntimeError("Checkpoint class ordering mismatch.")
    if payload["dataset_signature"] != DATASET_SIGNATURE:
        raise RuntimeError("Checkpoint dataset signature mismatch.")
    model.load_state_dict(payload["model_state"], strict=True)
    return payload


def train_phase(model, phase, epochs, optimizer, scheduler, patience, best_macro_f1, checkpoint_path):
    history_rows = []
    stale_epochs = 0

    for epoch in range(1, epochs + 1):
        train_metrics = run_epoch(model, train_loader, optimizer=optimizer)
        val_metrics = run_epoch(model, val_loader)
        scheduler.step(val_metrics["loss"])

        row = {
            "phase": phase,
            "epoch": epoch,
            **{f"train_{key}": value for key, value in train_metrics.items()},
            **{f"val_{key}": value for key, value in val_metrics.items()},
        }
        history_rows.append(row)

        print(
            f"{phase} {epoch:02d}/{epochs} | "
            f"train loss={train_metrics['loss']:.4f} f1={train_metrics['macro_f1']:.4f} | "
            f"val loss={val_metrics['loss']:.4f} f1={val_metrics['macro_f1']:.4f}"
        )

        if val_metrics["macro_f1"] > best_macro_f1 + 1e-6:
            best_macro_f1 = val_metrics["macro_f1"]
            stale_epochs = 0
            torch.save(checkpoint_payload(model, phase, epoch, best_macro_f1), checkpoint_path)
        else:
            stale_epochs += 1
            if stale_epochs >= patience:
                print(f"Early stopping {phase}; global best macro-F1={best_macro_f1:.4f}")
                break

    return history_rows, best_macro_f1


history = []
print("Training utilities are ready.")


## 7. Phase one ? classifier head


In [ ]:
seed_everything(SEED)
freeze_backbone(model)
optimizer_phase1 = optim.AdamW(
    [parameter for parameter in model.parameters() if parameter.requires_grad],
    lr=1e-3,
    weight_decay=1e-4,
)
scheduler_phase1 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_phase1,
    mode="min",
    factor=0.3,
    patience=2,
)

phase1_history, global_best_macro_f1 = train_phase(
    model=model,
    phase="phase1_head",
    epochs=10,
    optimizer=optimizer_phase1,
    scheduler=scheduler_phase1,
    patience=3,
    best_macro_f1=-1.0,
    checkpoint_path=PHASE1_PATH,
)
history.extend(phase1_history)
load_checkpoint(PHASE1_PATH, model)
shutil.copy2(PHASE1_PATH, CHAMPION_PATH)

print(f"Phase-one champion macro-F1: {global_best_macro_f1:.4f}")


## 8. Phase two ? selective fine-tuning

The phase-one champion remains the final champion unless fine-tuning improves validation macro-F1.


In [ ]:
load_checkpoint(CHAMPION_PATH, model)
unfreeze_final_blocks(model, block_count=2)
_, trainable_params = parameter_summary(model)
print(f"Trainable phase 2: {trainable_params:,}")

optimizer_phase2 = optim.AdamW(
    [parameter for parameter in model.parameters() if parameter.requires_grad],
    lr=1e-5,
    weight_decay=1e-4,
)
scheduler_phase2 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_phase2,
    mode="min",
    factor=0.3,
    patience=1,
)

phase2_history, global_best_macro_f1 = train_phase(
    model=model,
    phase="phase2_finetune",
    epochs=5,
    optimizer=optimizer_phase2,
    scheduler=scheduler_phase2,
    patience=2,
    best_macro_f1=global_best_macro_f1,
    checkpoint_path=CHAMPION_PATH,
)
history.extend(phase2_history)
champion_payload = load_checkpoint(CHAMPION_PATH, model)

history_path = REPORT_DIR / "potato_training_history.json"
history_path.write_text(json.dumps(history, indent=2), encoding="utf-8")
print(f"Global champion: {champion_payload['phase']} epoch {champion_payload['epoch']}")
print(f"Validation macro-F1: {champion_payload['best_val_macro_f1']:.4f}")


## 9. Training curves


In [ ]:
epoch_axis = np.arange(1, len(history) + 1)
train_loss = [row["train_loss"] for row in history]
val_loss = [row["val_loss"] for row in history]
train_f1 = [row["train_macro_f1"] for row in history]
val_f1 = [row["val_macro_f1"] for row in history]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(epoch_axis, train_loss, marker="o", label="Train")
axes[0].plot(epoch_axis, val_loss, marker="o", label="Validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Executed epoch")
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(epoch_axis, train_f1, marker="o", label="Train")
axes[1].plot(epoch_axis, val_f1, marker="o", label="Validation")
axes[1].set_title("Macro-F1")
axes[1].set_xlabel("Executed epoch")
axes[1].legend()
axes[1].grid(alpha=0.25)

plt.tight_layout()
plot_path = REPORT_DIR / "potato_training_curves.png"
plt.savefig(plot_path, dpi=160, bbox_inches="tight")
plt.show()
print(f"Saved: {plot_path}")


## 10. Validation calibration and rejection policy

Temperature and uncertainty thresholds are fitted on validation predictions only. They are locked before the test set is evaluated.


In [ ]:
def collect_logits(model, loader):
    model.eval()
    logits_parts, target_parts = [], []
    elapsed = 0.0
    with torch.inference_mode():
        for images, targets in loader:
            images = images.to(DEVICE, non_blocking=True)
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            start = time.perf_counter()
            logits = model(images)
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            elapsed += time.perf_counter() - start
            logits_parts.append(logits.cpu())
            target_parts.append(targets.cpu())
    return torch.cat(logits_parts), torch.cat(target_parts), elapsed


class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_temperature = nn.Parameter(torch.zeros(1))

    @property
    def temperature(self):
        return self.log_temperature.exp().clamp(0.05, 10.0)

    def forward(self, logits):
        return logits / self.temperature


def fit_temperature(logits, targets):
    scaler = TemperatureScaler()
    objective = nn.CrossEntropyLoss()
    optimizer = optim.LBFGS(scaler.parameters(), lr=0.05, max_iter=100)

    def closure():
        optimizer.zero_grad()
        loss = objective(scaler(logits), targets)
        loss.backward()
        return loss

    optimizer.step(closure)
    return float(scaler.temperature.detach().item())


def fit_rejection_policy(probabilities, targets, minimum_coverage=0.60):
    predictions = probabilities.argmax(axis=1)
    sorted_probs = np.sort(probabilities, axis=1)
    confidence = sorted_probs[:, -1]
    margin = sorted_probs[:, -1] - sorted_probs[:, -2]
    candidates = []

    for confidence_threshold in np.arange(0.50, 0.991, 0.01):
        for margin_threshold in np.arange(0.00, 0.501, 0.02):
            accepted = (confidence >= confidence_threshold) & (margin >= margin_threshold)
            coverage = accepted.mean()
            if coverage < minimum_coverage or accepted.sum() < num_classes:
                continue
            macro_f1 = f1_score(targets[accepted], predictions[accepted], average="macro", zero_division=0)
            accuracy = accuracy_score(targets[accepted], predictions[accepted])
            candidates.append((macro_f1, coverage, accuracy, confidence_threshold, margin_threshold))

    if not candidates:
        raise RuntimeError("Could not fit a rejection policy at the required validation coverage.")

    macro_f1, coverage, accuracy, confidence_threshold, margin_threshold = max(candidates)
    return {
        "confidence_threshold": float(confidence_threshold),
        "margin_threshold": float(margin_threshold),
        "validation_coverage": float(coverage),
        "validation_accepted_macro_f1": float(macro_f1),
        "validation_accepted_accuracy": float(accuracy),
    }


load_checkpoint(CHAMPION_PATH, model)
val_logits, val_targets_tensor, _ = collect_logits(model, val_loader)
temperature = fit_temperature(val_logits, val_targets_tensor)
val_probabilities = torch.softmax(val_logits / temperature, dim=1).numpy()
val_targets = val_targets_tensor.numpy()
rejection_policy = fit_rejection_policy(val_probabilities, val_targets)

calibration_payload = {
    "temperature": temperature,
    "rejection_policy": rejection_policy,
    "fitted_on": "validation",
}
calibration_path = MODEL_DIR / "potato_calibration.json"
calibration_path.write_text(json.dumps(calibration_payload, indent=2), encoding="utf-8")

print(json.dumps(calibration_payload, indent=2))


## 11. Locked single-pass test evaluation

Run this cell once after every previous decision is fixed. Test predictions do not select checkpoints, temperatures, thresholds or training settings.


In [ ]:
def expected_calibration_error(probabilities, targets, bins=15):
    confidence = probabilities.max(axis=1)
    predictions = probabilities.argmax(axis=1)
    edges = np.linspace(0.0, 1.0, bins + 1)
    ece = 0.0
    for lower, upper in zip(edges[:-1], edges[1:]):
        mask = (confidence > lower) & (confidence <= upper)
        if not mask.any():
            continue
        bin_accuracy = (predictions[mask] == targets[mask]).mean()
        ece += mask.mean() * abs(bin_accuracy - confidence[mask].mean())
    return float(ece)


def json_safe(value):
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if isinstance(value, np.generic):
        return value.item()
    return value


test_logits, test_targets_tensor, inference_seconds = collect_logits(model, test_loader)
test_targets = test_targets_tensor.numpy()
raw_probabilities = torch.softmax(test_logits, dim=1).numpy()
calibrated_probabilities = torch.softmax(test_logits / temperature, dim=1).numpy()
test_predictions = calibrated_probabilities.argmax(axis=1)

confidence = calibrated_probabilities.max(axis=1)
sorted_probabilities = np.sort(calibrated_probabilities, axis=1)
margin = sorted_probabilities[:, -1] - sorted_probabilities[:, -2]
accepted = (
    (confidence >= rejection_policy["confidence_threshold"])
    & (margin >= rejection_policy["margin_threshold"])
)

report = {
    "crop": "potato",
    "architecture": ARCHITECTURE,
    "checkpoint_phase": champion_payload["phase"],
    "test_size": int(len(test_targets)),
    "single_pass": True,
    "tta": False,
    "accuracy": accuracy_score(test_targets, test_predictions),
    "balanced_accuracy": balanced_accuracy_score(test_targets, test_predictions),
    "macro_f1": f1_score(test_targets, test_predictions, average="macro", zero_division=0),
    "weighted_f1": f1_score(test_targets, test_predictions, average="weighted", zero_division=0),
    "raw_nll": log_loss(test_targets, raw_probabilities, labels=list(range(num_classes))),
    "calibrated_nll": log_loss(test_targets, calibrated_probabilities, labels=list(range(num_classes))),
    "raw_ece": expected_calibration_error(raw_probabilities, test_targets),
    "calibrated_ece": expected_calibration_error(calibrated_probabilities, test_targets),
    "coverage": accepted.mean(),
    "accepted_accuracy": accuracy_score(test_targets[accepted], test_predictions[accepted]) if accepted.any() else None,
    "accepted_macro_f1": f1_score(test_targets[accepted], test_predictions[accepted], average="macro", zero_division=0) if accepted.any() else None,
    "mean_inference_ms_per_image": inference_seconds * 1000.0 / len(test_targets),
    "classification_report": classification_report(
        test_targets,
        test_predictions,
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    ),
}

report_path = REPORT_DIR / "potato_test_report.json"
report_path.write_text(json.dumps(json_safe(report), indent=2), encoding="utf-8")

matrix = confusion_matrix(test_targets, test_predictions, labels=list(range(num_classes)))
fig, ax = plt.subplots(figsize=(7, 6))
image = ax.imshow(matrix, cmap="Blues")
for row in range(matrix.shape[0]):
    for column in range(matrix.shape[1]):
        ax.text(column, row, matrix[row, column], ha="center", va="center")
ax.set_xticks(range(num_classes), class_names, rotation=25, ha="right")
ax.set_yticks(range(num_classes), class_names)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Potato ? Corrected Single-Pass Test Confusion Matrix")
fig.colorbar(image, ax=ax)
plt.tight_layout()
confusion_path = REPORT_DIR / "potato_confusion_matrix.png"
plt.savefig(confusion_path, dpi=160, bbox_inches="tight")
plt.show()

print(json.dumps(json_safe({key: value for key, value in report.items() if key != "classification_report"}), indent=2))
print(f"Saved report: {report_path}")


## 12. Safe calibrated inference

The production-facing result can be `ok` or `uncertain`. A low-confidence image never receives treatment advice automatically.


In [ ]:
def load_rgb_image(image_source):
    if isinstance(image_source, Image.Image):
        return image_source.convert("RGB")
    path = Path(image_source)
    if not path.exists() or not path.is_file():
        raise FileNotFoundError(f"Image not found: {path}")
    with Image.open(path) as image:
        return image.convert("RGB")


def predict_potato(image_source):
    image = load_rgb_image(image_source)
    tensor = eval_transform(image).unsqueeze(0).to(DEVICE)

    model.eval()
    with torch.inference_mode():
        logits = model(tensor) / temperature
        probabilities = torch.softmax(logits, dim=1)[0].cpu().numpy()

    order = np.argsort(probabilities)[::-1]
    predicted_index = int(order[0])
    confidence = float(probabilities[order[0]])
    margin = float(probabilities[order[0]] - probabilities[order[1]])
    accepted = (
        confidence >= rejection_policy["confidence_threshold"]
        and margin >= rejection_policy["margin_threshold"]
    )

    return {
        "status": "ok" if accepted else "uncertain",
        "crop": "potato",
        "disease": class_names[predicted_index] if accepted else None,
        "candidate_disease": class_names[predicted_index],
        "confidence": confidence,
        "top_two_margin": margin,
        "top_predictions": [
            {"class": class_names[int(index)], "probability": float(probabilities[index])}
            for index in order[: min(3, len(order))]
        ],
        "model": ARCHITECTURE,
        "single_pass": True,
    }


# Example after setting a real path:
# result = predict_potato("path/to/potato_leaf.jpg")
# print(json.dumps(result, indent=2))
print("predict_potato() is ready.")


## 13. Explainability and severity boundary

Grad-CAM++ may be attached to an accepted prediction to show where EfficientNet-B0 focused. It must be labelled **model attention**, not infected-area segmentation.

Numerical severity is deliberately excluded from this corrected classifier notebook. It will be implemented as a separately validated evidence-fusion pipeline using leaf masking, lesion evidence, attention localization, reliability checks and expert-labelled severity samples. Until that validation exists, the inference contract must not claim a medically or agriculturally validated spread percentage.


In [ ]:
def resolve_gradcam_target_layer(model):
    return model.features[-1]


print(f"Grad-CAM++ target layer: {resolve_gradcam_target_layer(model).__class__.__name__}")
print("Severity status: intentionally deferred to the validated severity pipeline.")


## Completion criteria

This notebook is complete only when it runs from a clean kernel, creates a reusable split manifest, retains the best checkpoint across both phases, fits calibration on validation data, evaluates the test set with one forward pass, and produces a safe uncertainty-aware inference result.

Before Android integration, the exported preprocessing specification, class order and golden-image logits must be checked against the mobile runtime.
